# Phase 8: resolution test results, C6 at 256 against 512 (half A, no GPU)

**CRISP-DM stage:** Evaluation

The question: **does a sharper photo help the inspector find small defects, and what does it cost in time and memory?** Only the image size changed between the two runs (see `08_resolution_C6`). This notebook reads the saved results of both and compares them.

1. A table of every benchmark metric at both sizes, with runtime per photo, peak memory and fitting time.
2. A paired bootstrap of the difference, 512 minus 256, the same as in Phases 5 and 7: the parts are drawn, and both sizes are measured on the same drawn parts.
3. Part by part: is each defective part ranked above every good photo more often at 512? This checks the small Sheet Metal defects (000, 005, 012) that Phases 6 and 7 found were missed.
4. A scatter plot of accuracy against runtime per photo, for all eight configurations and C6 at 512.

**Half B is not read. Development numbers, not final results.**

**Runs on:** the laptop.

In [ ]:
import csv
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Finds the project folder, whether the notebook runs from the project or from notebooks/
PROJECT_FOLDER = Path.cwd()
if not (PROJECT_FOLDER / "src").exists():
    PROJECT_FOLDER = PROJECT_FOLDER.parent
sys.path.insert(0, str(PROJECT_FOLDER))
from src import evaluate

RESULTS = PROJECT_FOLDER / "outputs/results"
# Where each run's files are: the 256 runs from Phases 5 and 7, and the 512 run from Phase 8
RUN_FOLDERS = {"C1": RESULTS / "05_core/C1", "C2": RESULTS / "05_core/C2", "C3": RESULTS / "07_core/C3",
               "C4": RESULTS / "05_core/C4", "C5": RESULTS / "05_core/C5", "C6": RESULTS / "07_core/C6",
               "C7": RESULTS / "07_core/C7", "C8": RESULTS / "07_core/C8", "C6_512": RESULTS / "08_resolution/C6_512"}
OUTPUT_FOLDER = RESULTS / "08_resolution"
FIGURE_FOLDER = PROJECT_FOLDER / "outputs/figures"
PRODUCTS = ["sheet_metal", "vial"]
METRICS = ["au_pro_5_percent", "au_pro_30_percent", "pixel_f1", "image_f1", "image_au_roc",
           "seconds_per_photo", "peak_gpu_gb", "fit_seconds", "peak_gpu_gb_fit"]
FIGURE_DPI = 150


def load_run(run_id, product):
    """Return each seed's metrics, scores and threshold, plus the labels and file names."""
    metric_list = []
    score_lists = []
    thresholds = []
    labels = None
    file_names = None
    for metric_file in sorted(RUN_FOLDERS[run_id].glob(run_id + "_" + product + "_seed*_metrics.json")):
        with open(metric_file, encoding="utf-8") as file:
            metrics = json.load(file)
        metric_list.append(metrics)
        thresholds.append(float(metrics["threshold"]))
        scores = []
        these_labels = []
        these_names = []
        with open(str(metric_file).replace("_metrics.json", "_photo_scores.csv"), newline="") as file:
            for photo in csv.DictReader(file):
                scores.append(float(photo["score"]))
                these_labels.append(photo["label"] == "bad")
                these_names.append(photo["file_name"])
        score_lists.append(torch.tensor(scores))
        labels = torch.tensor(these_labels)
        file_names = these_names
    return metric_list, score_lists, thresholds, labels, file_names


def mean_and_spread(values):
    """Return the mean, and the gap between the highest and lowest seed."""
    numbers = []
    for value in values:
        numbers.append(float(value))
    return sum(numbers) / len(numbers), max(numbers) - min(numbers)

## 1. Table: 256 against 512

Each value is the mean over 3 seeds, with the spread (highest minus lowest seed) in brackets.

In [ ]:
table_rows = []
for product in PRODUCTS:
    for run_id in ["C6", "C6_512"]:
        metric_list, score_lists, thresholds, labels, file_names = load_run(run_id, product)
        row = {"product": product, "run": run_id, "image_size": metric_list[0]["image_size"], "seeds": len(metric_list)}
        for metric in METRICS:
            values = []
            for metrics in metric_list:
                values.append(metrics[metric])
            mean, spread = mean_and_spread(values)
            row[metric] = round(mean, 4)
            row[metric + "_spread"] = round(spread, 4)
        table_rows.append(row)

with open(OUTPUT_FOLDER / "resolution_table.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(table_rows[0].keys()))
    writer.writeheader()
    for row in table_rows:
        writer.writerow(row)

for row in table_rows:
    print("===", row["product"], row["run"], "at", row["image_size"], "pixels")
    for metric in METRICS:
        print("  ", metric.ljust(20), row[metric], "(" + str(row[metric + "_spread"]) + ")")

## 2. Paired difference, 512 minus 256

A difference is called real only if its 95 percent interval leaves out 0. With 2 or 3 good parts per product, the intervals are wide.

In [ ]:
difference_rows = []
for product in PRODUCTS:
    ignored, scores_512, thresholds_512, labels, names_512 = load_run("C6_512", product)
    ignored, scores_256, thresholds_256, labels_256, names_256 = load_run("C6", product)
    # Both sizes must have scored the same photos in the same order
    if names_512 != names_256 or not torch.equal(labels, labels_256):
        raise ValueError("Photos differ between the two sizes for " + product)
    gap = evaluate.paired_bootstrap_difference((scores_512, thresholds_512), (scores_256, thresholds_256), labels, names_512)
    gap["product"] = product
    difference_rows.append(gap)
    for metric in ["image_f1", "image_au_roc"]:
        low = gap[metric + "_difference_low"]
        high = gap[metric + "_difference_high"]
        verdict = "not real"
        if low > 0 or high < 0:
            verdict = "real"
        print(product, "|", metric, "512 minus 256:", round(gap[metric + "_difference"], 3),
              "[" + str(round(low, 3)) + " to " + str(round(high, 3)) + "]", verdict)

with open(OUTPUT_FOLDER / "resolution_paired_differences.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(difference_rows[0].keys()))
    writer.writeheader()
    for row in difference_rows:
        writer.writerow(row)

## 3. Part by part: which defective parts are found more often at 512?

For each defective part, the share of its photos (all lightings) that score above **every** good photo of the same lighting. This needs no threshold, and it checks ranking within a lighting, so lighting drift does not decide it. Averaged over the 3 seeds.

In [ ]:
def lighting_of(file_name):
    """Return the lighting part of a name like 007_shift_2.png, here shift_2."""
    stem = Path(file_name).stem
    return stem[stem.index("_") + 1:]


part_rows = []
for product in PRODUCTS:
    for run_id in ["C6", "C6_512"]:
        ignored, score_lists, thresholds, labels, file_names = load_run(run_id, product)
        found = {}
        for scores in score_lists:
            # The highest good score under each lighting
            highest_good = {}
            for position in range(len(file_names)):
                if not labels[position]:
                    lighting = lighting_of(file_names[position])
                    value = float(scores[position])
                    if lighting not in highest_good or value > highest_good[lighting]:
                        highest_good[lighting] = value
            for position in range(len(file_names)):
                if labels[position]:
                    part = evaluate.part_number(file_names[position])
                    if part not in found:
                        found[part] = []
                    above = float(scores[position]) > highest_good[lighting_of(file_names[position])]
                    found[part].append(above)
        for part in sorted(found):
            share = sum(found[part]) / len(found[part])
            part_rows.append({"product": product, "run": run_id, "part": part, "share_above_every_good": round(share, 3)})

with open(OUTPUT_FOLDER / "resolution_by_part.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(part_rows[0].keys()))
    writer.writeheader()
    for row in part_rows:
        writer.writerow(row)

for product in PRODUCTS:
    print("===", product, "| part: share at 256 -> share at 512")
    for row in part_rows:
        if row["product"] == product and row["run"] == "C6":
            for other in part_rows:
                if other["product"] == product and other["run"] == "C6_512" and other["part"] == row["part"]:
                    print("  ", row["part"], ":", row["share_above_every_good"], "->", other["share_above_every_good"])

## 4. Accuracy against runtime per photo

Each point is one configuration, averaged over its seeds. The runtime axis is logarithmic, because the slowest configuration is about 30 times slower than the fastest. The arrow shows the step from C6 at 256 to C6 at 512. Runtime is the model's time on a Tesla T4, without reading photos from disk.

In [ ]:
RUN_IDS = ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C6_512"]
LABELS = {"C6_512": "C6 at 512"}
# The same colours as the Phase 7 chart, with C6 at 512 drawn in C6's colour
COLOURS = {"C1": "#2a78d6", "C2": "#eb6834", "C3": "#56b4e9", "C4": "#1baf7a", "C5": "#eda100",
           "C6": "#cc79a7", "C7": "#7a5195", "C8": "#8c6d31", "C6_512": "#cc79a7"}
ACCURACY_MEASURES = [("image_au_roc", "Image AU-ROC"), ("au_pro_30_percent", "AU-PRO 30%")]

figure, axes = plt.subplots(len(ACCURACY_MEASURES), len(PRODUCTS), figsize=(11, 8.5), squeeze=False)
for column in range(len(PRODUCTS)):
    product = PRODUCTS[column]
    for row_number in range(len(ACCURACY_MEASURES)):
        measure, label = ACCURACY_MEASURES[row_number]
        axis = axes[row_number][column]
        points = {}
        for run_id in RUN_IDS:
            metric_list = load_run(run_id, product)[0]
            runtime = []
            accuracy = []
            for metrics in metric_list:
                runtime.append(metrics["seconds_per_photo"])
                accuracy.append(metrics[measure])
            points[run_id] = (mean_and_spread(runtime)[0], mean_and_spread(accuracy)[0])
        for run_id in RUN_IDS:
            x, y = points[run_id]
            marker = "o"
            if run_id == "C6_512":
                marker = "D"
            axis.scatter([x], [y], color=COLOURS[run_id], marker=marker, s=60, zorder=3)
            axis.annotate(LABELS.get(run_id, run_id), (x, y), textcoords="offset points", xytext=(6, 4), fontsize=9)
        axis.annotate("", xy=points["C6_512"], xytext=points["C6"],
                      arrowprops={"arrowstyle": "->", "color": COLOURS["C6"], "lw": 1.2})
        axis.set_xscale("log")
        axis.set_xlabel("Seconds per photo (log scale)")
        axis.set_ylabel(label)
        axis.set_ylim(0, 1)
        axis.set_title(product.replace("_", " ").title() + ", half A")
        axis.grid(alpha=0.3)
figure.suptitle("Accuracy against runtime. Development numbers on half A, not final results", y=1.0)
figure.tight_layout()
figure.savefig(FIGURE_FOLDER / "phase8_accuracy_vs_runtime.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()

## Both lines for C6 at 512, needed for the Lock

The Final test scores C6 at 512 on half B too, so both of its lines must be fixed now, on half A:
- **The benchmark line** was already set by the run itself, from the good validation photos.
- **The cost-based line** is found here with the same shared code as in Phases 6 and 7 (`src/cost.py`). It uses the project's main business setting: a defect rate of 1 percent and a missed defect costing 10 false rejects.

In [ ]:
import csv
import json
from src import cost

line_rows = []
for product in ["sheet_metal", "vial"]:
    for seed in ["42", "43", "44"]:
        run_name = "C6_512_" + product + "_seed" + seed
        labels = []
        scores = []
        with open(RUN_FOLDERS["C6_512"] / (run_name + "_photo_scores.csv"), encoding="utf-8") as file:
            for row in csv.DictReader(file):
                labels.append(row["label"] == "bad")
                scores.append(float(row["score"]))
        labels = torch.tensor(labels)
        scores = torch.tensor(scores)
        with open(RUN_FOLDERS["C6_512"] / (run_name + "_metrics.json"), encoding="utf-8") as file:
            benchmark_line = json.load(file)["threshold"]
        best = cost.cheapest_threshold(scores, labels, cost.MAIN_DEFECT_RATE, cost.MAIN_COST_RATIO)
        line_rows.append({"config": "C6_512", "product": product, "seed": seed,
                          "benchmark_threshold": round(benchmark_line, 4),
                          "cost_based_threshold": round(best["threshold"], 4),
                          "cost_at_cost_based_line": round(best["cost"], 4)})

with open(OUTPUT_FOLDER / "C6_512_thresholds.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(line_rows[0].keys()))
    writer.writeheader()
    for row in line_rows:
        writer.writerow(row)
for row in line_rows:
    print(row)